# Spectral Guidance Example
## Gaussian prior



In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from torch.distributions.multivariate_normal import MultivariateNormal
from torch.optim import Adam
from torch.optim.lr_scheduler import ExponentialLR
from diffusers import DDPMScheduler, DDIMScheduler

from spectral import whitening
from examples.encoder import TimeConditionedEncoder

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

axes_fontsize = 13
ticks_fontsize = 10
ticks_width = 0.5
plot_linewidth = 2.5
spines_linewidth = 0.5
plot_strengths = [10]
figsize = (3.5, 2.8)

device = "cuda:0" if torch.cuda.is_available() else "cpu"

In [ ]:
class Prior:
    def __init__(
        self,
        dim: int,
        rho: float = 0.7,
        lambda_1: float = 30.0,
        device: str = "cpu",
        generator = None,
    ):
        self.dim = dim
        self.device = device

        A = torch.randn(dim, dim, device=device, generator=generator)
        Q, _ = torch.linalg.qr(A)

        # Geometric eigenvalue decay: lambda_k = lambda_1 * rho^k, k = 0, ..., dim-1.
        eigenvalues = lambda_1 * rho ** torch.arange(dim, device=device, dtype=Q.dtype)

        self.eigenvalues = eigenvalues           # (D,), descending
        self.eigenvectors = Q                    # (D, D), columns are u_k

        cov = Q @ torch.diag(eigenvalues) @ Q.T
        cov = 0.5 * (cov + cov.T)

        self.dist = MultivariateNormal(
            loc=torch.zeros(dim, device=device),
            covariance_matrix=cov,
        )

    def sample(self, batch_size: int):
        return self.dist.sample((batch_size,))   # (B, D)
        
prior = Prior(dim=20, rho=0.7, lambda_1=40.0, device=device)
prior_samples = prior.sample(batch_size=100)
print(prior_samples.shape)

## Train $f_\phi$

In [ ]:
num_eigenfunctions = 3
num_train_steps = 5000
batch_size = 4096
lr = 1e-4
ridge = 1e-3
eps = 1e-10

noise_scheduler = DDIMScheduler(
    beta_schedule="linear",
    beta_start=1e-4,
    beta_end=0.02,
    num_train_timesteps=1000,
    clip_sample=False,
)
noise_scheduler.set_timesteps(1000)
timesteps = [int(t) for t in noise_scheduler.timesteps]

phi_encoder = TimeConditionedEncoder(in_dim=prior.dim, hidden_dim=64, t_dim=64, out_dim=num_eigenfunctions).to(device)
phi_encoder.train()

optimizer = Adam(phi_encoder.parameters(), lr=lr)
scheduler = ExponentialLR(optimizer, gamma=0.9995)

pbar = tqdm(range(num_train_steps), desc="Training phi network")

for train_step in pbar:
    x0 = prior.sample(batch_size).to(device)
    t = np.random.choice(timesteps)
    t_batch = torch.full((x0.shape[0],), t, device=device)
    noise_a = torch.randn_like(x0)
    noise_b = torch.randn_like(x0)
    
    x_a = noise_scheduler.add_noise(x0, noise_a, t_batch)
    x_b = noise_scheduler.add_noise(x0, noise_b, t_batch)
       
    with torch.no_grad():
        phi_a = phi_encoder(x_a, t_batch)
        
    phi_b = phi_encoder(x_b, t_batch)
    mu, W = whitening(phi_a, ridge=ridge)
    phi_a_w = (phi_a - mu) @ W
    phi_b_w = (phi_b - mu) @ W
    phi_a_w = phi_a_w / (eps + phi_a_w.std(dim=0, keepdims=True)) # Renormalize
    phi_b_w = phi_b_w / (eps + phi_b_w.std(dim=0, keepdims=True)) # Renormalize

    eigenvalues = (phi_a_w * phi_b_w).mean(0)
    loss = (1.0 - eigenvalues.mean())

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    scheduler.step()
    pbar.set_postfix({'lr' : f"{optimizer.param_groups[0]['lr']:1.3e}"})
    
phi_encoder.eval()
eigenvalues_t = {}
with torch.no_grad():
    for t in tqdm(timesteps):
        x0 = prior.sample(4096).to(device)
        t_batch = torch.full((x0.shape[0],), t, device=device)
        noise_a, noise_b = torch.randn_like(x0), torch.randn_like(x0)
        x_a = noise_scheduler.add_noise(x0, noise_a, t_batch)
        x_b = noise_scheduler.add_noise(x0, noise_b, t_batch)
        phi_a = phi_encoder(x_a, t_batch)
        phi_b = phi_encoder(x_b, t_batch)
        mu, W = whitening(phi_a, ridge=0.0)
        phi_a_w = (phi_a - mu) @ W
        phi_b_w = (phi_b - mu) @ W
        eigenvalues_t[t] = (phi_a_w * phi_b_w).sum(0).cpu() / 4096
        
eigenvalues_timestamps = sorted(eigenvalues_t.keys()) # (T,)
eigenvalues_stacked = torch.stack([torch.sort(eigenvalues_t[t])[0] for t in eigenvalues_timestamps]) # (T, K)

## Ground-Truth Spectrum

In [ ]:
alpha_bars = noise_scheduler.alphas_cumprod[noise_scheduler.timesteps].cpu()
gt_eigs = prior.eigenvalues[:num_eigenfunctions].cpu()

true_eigenvalues = torch.sort(((alpha_bars[:, None] * gt_eigs[None, :]) / (alpha_bars[:, None] * gt_eigs[None, :] + (1 - alpha_bars[:, None]))), dim=-1)[0]

cmap = plt.get_cmap("Set1")
colors = [cmap(i) for i in [2,1,0]]

fig, ax = plt.subplots(1, 1, figsize=figsize)
for k in range(num_eigenfunctions-1,-1,-1):
    ax.plot(
        noise_scheduler.timesteps,
        true_eigenvalues[:, k],
        color=colors[k],
        linewidth=2.5,
        label=f"$\lambda_{4-k}$",
    )
ax.set_xlabel("Diffusion time", fontsize=axes_fontsize)
ax.set_ylabel("Ground-truth eigenvalues", fontsize=axes_fontsize)
ax.tick_params(axis='both', which='major', labelsize=ticks_fontsize, width=ticks_width)
ax.tick_params(axis='both', which='major', labelsize=ticks_fontsize, width=ticks_width)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=axes_fontsize)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(spines_linewidth)  # adjust thickness
#fig.savefig("./gaussian_example_gt_eig.pdf", bbox_inches="tight", pad_inches=0)

## Estimated Spectrum

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=figsize)
for k in range(num_eigenfunctions-1,-1,-1):
    ax.plot(
        eigenvalues_timestamps,
        eigenvalues_stacked[:, k], 
        color=colors[k],
        linewidth=1.5,
        label=f"$\lambda_{4-k}$",
    )
ax.set_xlabel("Diffusion time", fontsize=axes_fontsize)
ax.set_ylabel("Estimated eigenvalues", fontsize=axes_fontsize)
ax.tick_params(axis='both', which='major', labelsize=ticks_fontsize, width=ticks_width)
ax.tick_params(axis='both', which='major', labelsize=ticks_fontsize, width=ticks_width)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=axes_fontsize)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(spines_linewidth)
#fig.savefig("./gaussian_example_estimated_eig.pdf", bbox_inches="tight", pad_inches=0)

## Eigenvalue Residuals

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=figsize)
for k in range(num_eigenfunctions-1,-1,-1):
    ax.plot(
        noise_scheduler.timesteps,
        (true_eigenvalues[:, k].flip(0) - eigenvalues_stacked[:, k]).abs(),
        color=colors[k],
        linewidth=1.0,
        label=f"$\lambda_{4-k}$",
    )
ax.set_xlabel("Diffusion time", fontsize=axes_fontsize)
ax.set_ylabel("Eigenvalue residuals", fontsize=axes_fontsize)
ax.tick_params(axis='both', which='major', labelsize=ticks_fontsize, width=ticks_width)
ax.tick_params(axis='both', which='major', labelsize=ticks_fontsize, width=ticks_width)
ax.legend(fontsize=axes_fontsize)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(spines_linewidth)  # adjust thickness
#fig.savefig("./gaussian_example_residuals.pdf", bbox_inches="tight", pad_inches=0)

## Principal Subspace Error

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=figsize)
n_samples = 4096
mean_cos = []
U_gt = prior.eigenvectors[:, :num_eigenfunctions].to(device)  # (D, K) ground truth

with torch.no_grad():
    for t in timesteps:
        x0 = prior.sample(n_samples).to(device)
        t_batch = torch.full((n_samples,), t, device=device)
        noise = torch.randn_like(x0)
        xt = noise_scheduler.add_noise(x0, noise, t_batch)  # (B, D)

        phi_out = phi_encoder(xt, t_batch)  # (B, K)

        # Least squares: find V in R^{D x K} such that xt @ V ~ phi_out
        # V = (xt^T xt)^{-1} xt^T phi_out
        xt_c = xt - xt.mean(0)
        phi_c = phi_out - phi_out.mean(0)
        V, _, _, _ = torch.linalg.lstsq(xt_c, phi_c)  # (D, K)

        # Orthonormalize V before comparing subspaces
        V_orth, _ = torch.linalg.qr(V)  # (D, K)

        # Principal angles via SVD of U_gt^T V_orth
        M = U_gt.T @ V_orth  # (K, K)
        cos_angles = torch.linalg.svdvals(M)  # (K,), cosines of principal angles, 1=aligned

        mean_cos.append(cos_angles.cpu().mean().item())

ax.plot(timesteps, mean_cos, linewidth=1, color="black")
ax.set_ylabel("Cosine principal angles", fontsize=axes_fontsize)
ax.set_xlabel("Diffusion Time", fontsize=axes_fontsize)
ax.tick_params(axis='both', which='major', labelsize=ticks_fontsize, width=ticks_width)
ax.tick_params(axis='both', which='major', labelsize=ticks_fontsize, width=ticks_width)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(spines_linewidth)  # adjust thickness
    
plt.show()
#fig.savefig("./gaussian_example_mean_cos.pdf", bbox_inches="tight", pad_inches=0)